# Credit Card Fraud Pattern Analysis

How do you spot the 0.172% of transactions that are fraud, without drowning a fraud team in false alarms?

**Dataset:** Credit Card Fraud Detection on Kaggle ([mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)), collected by Worldline and the Machine Learning Group of ULB. Transactions by European cardholders in September 2013, over two days. Features V1–V28 are PCA-anonymised, so no real customer behaviour can be read from them. The data covers only two days in 2013, so it cannot represent modern fraud patterns.

**Tools:** Python, pandas, scikit-learn, matplotlib. SQL version of these questions: `sql/fraud_queries.sql`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("../data/creditcard.csv")
print(df.shape)
print(df.dtypes.value_counts())

## 1. How imbalanced is this data?

Expected result from the published dataset: **284,807 transactions, 492 fraudulent (0.172%), 284,315 legitimate** — about 1 in 579 transactions is fraud.

In [ ]:
counts = df["Class"].value_counts()
fraud_rate = 100 * counts.get(1, 0) / len(df)
print(counts)
print(f"Fraud rate: {fraud_rate:.3f}%")
print(f"About 1 in {round(len(df) / counts.get(1, 1)):,} transactions is fraud.")

counts.plot(kind="bar", color=["steelblue", "firebrick"])
plt.title("Transactions by class (0 = legitimate, 1 = fraud)")
plt.ylabel("Transactions")
plt.xticks(rotation=0)
plt.show()

## 2. The accuracy trap

If you predict every transaction as legitimate, you get **99.83% accuracy** — and detect zero fraud. Accuracy on its own is meaningless when one class is 0.172% of the data. That is why this project judges models on precision, recall, F1 and the area under the precision–recall curve (AUPRC), which is what the Kaggle page itself recommends.

In [ ]:
baseline_accuracy = 100 * (len(df) - df["Class"].sum()) / len(df)
print(f"Accuracy of predicting everything as legitimate: {baseline_accuracy:.2f}%")
print(f"Fraud detected by that rule: 0 out of {int(df['Class'].sum())}")

## 3. Amount: fraud vs legitimate

Do fraudulent transactions look different in size? Compare the distribution of `Amount` for the two classes, and the fraud rate inside amount bands.

In [ ]:
print(df.groupby("Class")["Amount"].describe().round(2))

df.boxplot(column="Amount", by="Class")
plt.title("Transaction amount by class")
plt.suptitle("")
plt.show()

bins = [0, 10, 50, 100, 500, float("inf")]
labels = ["Under $10", "$10-$50", "$50-$100", "$100-$500", "$500+"]
df["amount_band"] = pd.cut(df["Amount"], bins=bins, labels=labels)
band = df.groupby("amount_band", observed=True).agg(
    transactions=("Class", "size"),
    fraud=("Class", "sum"),
)
band["fraud_rate_pct"] = 100 * band["fraud"] / band["transactions"]
print(band.round(3))

## 4. When does fraud happen?

`Time` is seconds since the first transaction in the file, so hour of day is `(Time // 3600) % 24`. This shows whether fraud clusters at particular hours.

In [ ]:
df["hour"] = (df["Time"] // 3600 % 24).astype(int)
hourly = df.groupby("hour").agg(
    transactions=("Class", "size"),
    fraud=("Class", "sum"),
)
hourly["fraud_rate_pct"] = 100 * hourly["fraud"] / hourly["transactions"]
print(hourly.round(3))

hourly["fraud"].plot(kind="bar", color="firebrick")
plt.title("Fraud transactions by hour of day")
plt.xlabel("Hour")
plt.ylabel("Fraud transactions")
plt.show()

## 5. A simple baseline model

Logistic regression on the anonymised features plus scaled `Amount`, with `class_weight='balanced'` so the 492 fraud cases are not ignored. Evaluated on a held-out test split with precision, recall, F1 and AUPRC — never accuracy alone.

For reference, in one published run of this dataset logistic regression reached an average precision of 0.704 and ROC-AUC of 0.973, and a random forest reached average precision 0.838 and ROC-AUC 0.957. Your numbers will differ slightly with a different random split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = [c for c in df.columns if c.startswith("V")] + ["Amount"]
X = df[features].copy()
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
proba = model.predict_proba(X_test_s)[:, 1]
pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, pred, digits=3))
print(f"Average precision (AUPRC): {average_precision_score(y_test, proba):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.3f}")

## 6. What the metrics mean for a fraud team

- **Recall** is the share of real fraud the model catches. Low recall means fraud slips through.
- **Precision** is the share of flagged transactions that are really fraud. Low precision means analysts waste time on false alarms.
- **F1** balances the two. **AUPRC** summarises the precision–recall trade-off in one number and is the honest single score for data this imbalanced.
- Raising recall usually means more false alarms. A real fraud team picks a threshold by cost: what does a missed fraud cost vs what does a false alarm cost in analyst time and customer friction?

## Limitations

Two days of data from September 2013, anonymised features, and no customer history. This is a pattern analysis on a public sample — not a production fraud system.